In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Carga de la tabla drivers

drivers = pd.read_csv(
    "../data/drivers.csv",
    sep=",",
    thousands=None,
    decimal=".",
)
drivers.head()

In [ ]:
# Carga de la tabla timesheet
timesheet = pd.read_csv(
    "../data/timesheet.csv",
    sep=",",
    thousands=None,
    decimal=".",
)
timesheet.head()

In [ ]:
# Media de las horas y las millas de cada conductor
mean_timesheet = timesheet.groupby("driverId")[["hours-logged", "miles-logged"]].mean()
mean_timesheet.head()

In [ ]:
# Registros con valores por debajo de la media del grupo
# transform("mean") repite la media del conductor en cada una de sus filas
timesheet_with_means = timesheet.copy()
timesheet_with_means["mean_hours-logged"] = timesheet.groupby("driverId")[
    "hours-logged"
].transform("mean")
timesheet_with_means.head()

In [ ]:
timesheet_below = timesheet_with_means[
    timesheet_with_means["hours-logged"] < timesheet_with_means["mean_hours-logged"]
]
display(timesheet_below.head(), timesheet_below.tail())

In [ ]:
# Cantidad total de horas y millas de cada conductor
sum_timesheet = timesheet.groupby("driverId", as_index=False)[
    ["hours-logged", "miles-logged"]
].sum()
sum_timesheet.head(10)

In [ ]:
# Valores mínimo y máximo de horas registradas por cada conductor
hours_range = timesheet.groupby("driverId")["hours-logged"].agg(["min", "max"])
hours_range

In [ ]:
# Se agrega el nombre de cada conductor (equivale al INNER JOIN de SQL)
summary = pd.merge(
    sum_timesheet,
    drivers[["driverId", "name"]],
    on="driverId",
)
summary

In [ ]:
# Almacenamiento de los resultados

summary.to_csv(
    "../submission/summary.csv",
    sep=",",
    header=True,
    index=False,
)

In [ ]:
# Los conductores con mayor cantidad total de millas registradas
top10 = summary.sort_values("miles-logged", ascending=False).head(10)
top10

In [ ]:
# Creación de un gráfico de barras horizontales
# La columna 'name' pasa a ser el nombre de las filas
top10 = top10.set_index("name")

# Paleta de colores:
#   tab:blue     tab:red       tab:pink
#   tab:orange   tab:purple    tab:gray
#   tab:green    tab:brown     tab:olive
#   tab:cyan
top10["miles-logged"].plot.barh(color="tab:orange", alpha=0.6)

plt.gca().invert_yaxis()

plt.gca().get_xaxis().set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ","))
)

plt.xticks(rotation=90)

plt.gca().spines["left"].set_color("lightgray")
plt.gca().spines["bottom"].set_color("gray")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.savefig("../submission/top10_drivers.png", bbox_inches="tight")